# Visualization — Evaluation Report

Standalone notebook for model evaluation visualization (same style as `05_evaluate_model.ipynb`).

**How to run:**
1. `Runtime → Run all`, **or**
2. If `rows_cache.json` already exists from STEP6, inference is skipped and cache is loaded automatically.

In [ ]:
import os
print("Begin 004...")
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_HF'] = '1'
print('Use HF = 1')
print('Set PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True')


## 0 · Install Dependencies

> After installation → **Restart session** before continuing.

In [ ]:
!pip install Levenshtein==0.26.1 plotnine==0.14.5 evaluate==0.4.4 cer==1.2.0 rouge_score==0.1.2 seaborn bitsandbytes python-dateutil --quiet


## 1 · Configuration

In [ ]:
from pathlib import Path
import re
import json
# Edit these paths to match your environment
BASE_DIR    = Path('/content/drive/MyDrive/INTERN-BIWOCO/sample-for-multi-modal-document-to-json-with-sagemaker-ai')



def is_valid_checkpoint(ckpt_path):
    cfg = ckpt_path / 'adapter_config.json'
    if not cfg.exists(): return False
    data = json.load(open(cfg))
    return 'language_model' not in data.get('target_modules', '')

CHECKPOINT = str(sorted(
    [p for p in (BASE_DIR / 'models/finetune').glob('*/checkpoint-*')
     if is_valid_checkpoint(p)],
    key=lambda p: p.stat().st_mtime
)[-1])


DATASET_DIR = BASE_DIR / 'data' / 'swift_dataset'
IMAGES_DIR  = DATASET_DIR / 'images'
MODEL_NAME  = Path(CHECKPOINT).parent.name   # used as label in charts

CACHE_FILE  = BASE_DIR / 'rows_cache.json'

print(f"BASE_DIR   : {BASE_DIR}")
print(f"CHECKPOINT : {CHECKPOINT}")
print(f"MODEL_NAME : {MODEL_NAME}")
print(f"Cache      : {'exists' if CACHE_FILE.exists() else 'NOT found — will run inference'}")

In [ ]:
# Clear VRAM before inference (essential after training on T4 15GB)
import torch, gc

for var in ['model', 'trainer', 'engine', 'optimizer']:
    if var in dir():
        del globals()[var]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.mem_get_info()[1] / 1024**3
print(f'VRAM free: {free:.1f} GB / {total:.1f} GB')
if free < 8:
    print('WARNING: < 8 GB free — inference may OOM. Try Runtime > Restart & Run All.')

## 2 · Load Data

Loads from cache if available, otherwise runs inference and saves cache.

In [ ]:
import json, torch, gc
from pathlib import Path

def load_or_infer():
    if CACHE_FILE.exists():
        print(f"Loading from cache: {CACHE_FILE}")
        with open(CACHE_FILE) as f:
            return json.load(f)

    print("No cache found — running inference...")
    from swift import TransformersEngine, RequestConfig, InferRequest
    from tqdm import tqdm

    with open(DATASET_DIR / 'conversations_test_swift_format.json') as f:
        test_data = json.load(f)

    engine = TransformersEngine(
        'Qwen/Qwen2-VL-2B-Instruct',
        adapters=[CHECKPOINT],
        max_pixels=200704,
        quantization_bit=4,
        torch_dtype='bfloat16',
        max_lenght='1850'
    )
    req_cfg = RequestConfig(max_tokens=2048, temperature=0)

    def _safe(v):
        if v is None: return ''
        if isinstance(v, list): return json.dumps(v, ensure_ascii=False)
        return str(v).strip()

    # def predict(sample):
    #     imgs = [str(IMAGES_DIR / Path(p).name) for p in sample['images']]
    #     req  = InferRequest(messages=sample['messages'][:2], images=imgs)
    #     out  = engine.infer([req], req_cfg)[0].choices[0].message.content.strip()
    #     out  = out.strip('`').removeprefix('json').strip()
    #     try:    return json.loads(out), True
    #     except: return {}, False

    hints_by_type = {

        "AUS_DRIVER_LICENSE": "",
        "AUS_PASSPORT":       "",
        "AUS_MEDICARE_CARD":  "Pay attention to: cardholders is a list",
        "AUS_ENERGY_BILL":    ".",
        "AUS_WWC_CARD":       "Pay attention to: wwc_type is Employee or Volunteer, wwc_number have format AAAAAAA-AA",
    }

    def predict(sample):
        imgs = [str(IMAGES_DIR / Path(p).name) for p in sample['images']]
        
        gt       = json.loads(sample['messages'][2]['content'])
        doc_type = gt.get('document_type', '')
        schema_str = json.dumps(gt, indent=2, ensure_ascii=False)
        hint = hints_by_type.get(doc_type, "")

        user_prompt = f"""Extract all fields from this {doc_type} document.
    {hint}

    Return ONLY valid JSON matching this structure (keys must match exactly):
    {schema_str}

    Rules:
    - null for missing fields, do NOT omit keys
    - Dates: YYYY-MM-DD
    - No markdown, no explanation"""

        msgs = [
            sample['messages'][0],
            {"role": "user", "content": user_prompt}
        ]

        req = InferRequest(messages=msgs, images=imgs)
        out = engine.infer([req], req_cfg)[0].choices[0].message.content.strip()
        out = out.strip('`').removeprefix('json').strip()
        try:    return json.loads(out), True
        except: return {}, False
        



    rows = []
    for s in tqdm(test_data, desc='Inference'):
        gt       = json.loads(s['messages'][2]['content'])
        pred, ok = predict(s)
        rows.append({'image': s['images'][0],
                     'doc_type': gt.get('document_type'),
                     'gt': gt, 'pred': pred, 'valid_json': ok})

    del engine; torch.cuda.empty_cache(); gc.collect()

    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"Cache saved to {CACHE_FILE}")
    return rows

rows = load_or_infer()
print(f"\nTotal samples : {len(rows)}")
print(f"Valid JSON    : {sum(r['valid_json'] for r in rows)}/{len(rows)}")

### (Optional) Save `rows` from STEP6 to cache

In [ ]:
# If you already have `rows` from STEP6, uncomment to save:
# with open(CACHE_FILE, 'w', encoding='utf-8') as f:
#     json.dump(rows, f, ensure_ascii=False, indent=2)
# print(f"Saved {len(rows)} rows to {CACHE_FILE}")

## 2.5 · Normalize Dates to YYYY-MM-DD

Converts all date values in both `gt` and `pred` to `YYYY-MM-DD` before any metric calculation.

Handles formats such as: `DD/MM/YYYY`, `MM/DD/YYYY`, `DD-MM-YYYY`, `YYYY/MM/DD`, `D MMM YYYY`, `YYYYMMDD`, etc.

In [ ]:
from dateutil import parser as dtparser
from dateutil.parser import ParserError
import re

# Fields whose names suggest they contain dates
DATE_KEYWORDS = ['date', 'ngay', 'day', 'dated', 'dob', 'birth', 'expiry', 'expire',
                 'issued', 'issue', 'valid', 'from', 'to', 'start', 'end', 'period']

def is_date_field(field_name: str) -> bool:
    name = field_name.lower()
    return any(kw in name for kw in DATE_KEYWORDS)

def try_parse_date(value: str) -> str:
    """Try to parse a string as a date and return YYYY-MM-DD, or original string on failure."""
    if not value or not isinstance(value, str):
        return value

    v = value.strip()
    if not v:
        return v

    # Skip if already YYYY-MM-DD
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}', v):
        return v

    # Common explicit patterns first (avoids dateutil ambiguity)
    patterns = [
        (r'^(\d{1,2})/(\d{1,2})/(\d{4})$',  '%d/%m/%Y'),   # DD/MM/YYYY
        (r'^(\d{4})/(\d{2})/(\d{2})$',        '%Y/%m/%d'),   # YYYY/MM/DD
        (r'^(\d{1,2})-(\d{1,2})-(\d{4})$',    '%d-%m-%Y'),   # DD-MM-YYYY
        (r'^(\d{8})$',                           '%Y%m%d'),     # YYYYMMDD
        (r'^(\d{4})(\d{2})(\d{2})$',           '%Y%m%d'),
    ]
    from datetime import datetime
    for pattern, fmt in patterns:
        if re.fullmatch(pattern, v):
            try:
                return datetime.strptime(v, fmt).strftime('%Y-%m-%d')
            except ValueError:
                pass

    # Fallback: dateutil with dayfirst=True
    try:
        return dtparser.parse(v, dayfirst=True).strftime('%Y-%m-%d')
    except (ParserError, OverflowError, ValueError):
        return v   # return original if not parseable

def normalize_dates(record: dict) -> dict:
    """Normalize all date fields in a dict to YYYY-MM-DD in-place."""
    for k, v in record.items():
        if is_date_field(k) and isinstance(v, str):
            record[k] = try_parse_date(v)
    return record

# Apply to both gt and pred in every row
n_converted = 0
for r in rows:
    before_gt   = json.dumps(r['gt'],   sort_keys=True)
    before_pred = json.dumps(r['pred'], sort_keys=True)

    normalize_dates(r['gt'])
    normalize_dates(r['pred'])

    if json.dumps(r['gt'], sort_keys=True) != before_gt or \
       json.dumps(r['pred'], sort_keys=True) != before_pred:
        n_converted += 1

print(f"Date normalization complete.")
print(f"Rows with at least one date field changed: {n_converted} / {len(rows)}")

# Preview a sample
date_fields = [f for r in rows for f in r['gt'] if is_date_field(f)]
print(f"\nDate fields detected: {sorted(set(date_fields))}")

## 3 · Build DataFrame (equivalent to `df_multi` in file 05)

`dist` encoding follows the same convention as file `05`:

| Value | Meaning |
|---|---|
| -1 | missing groundtruth |
| 0..6 | Levenshtein distance |
| -2 → 7 | predicted 'None' |
| -3 → 8 | key missing |
| -4 → 9 | invalid JSON |

In [ ]:
import pandas as pd
import Levenshtein

def _safe(v):
    if v is None: return ''
    if isinstance(v, list): return json.dumps(v, ensure_ascii=False)
    return str(v).strip()

def flatten_dict(d, parent_key='', sep='.'):
    """Flatten nested dict: {'front': {'holder': {'name': 'A'}}} 
       → {'front.holder.name': 'A'}"""
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        elif isinstance(v, list):
            items[new_key] = json.dumps(v, ensure_ascii=False)
        else:
            items[new_key] = v
    return items

EXCLUDE_FIELDS = {
    'back.barcode_number',
}

all_fields = sorted({k for r in rows for k in r['gt']} - EXCLUDE_FIELDS)
records = []

for r in rows:
    r['gt_flat']   = flatten_dict(r['gt'])
    r['pred_flat'] = flatten_dict(r['pred']) if r['valid_json'] else {}

all_fields = sorted({k for r in rows for k in r['gt_flat']} - EXCLUDE_FIELDS)

records = []

for r in rows:
    for field in all_fields:
        if field not in r['gt_flat']:
            continue
        gt_val = _safe(r['gt_flat'].get(field))

        if not r['valid_json']:
            pred_val = ''; dist = -4
        elif field not in r['pred_flat']:
            pred_val = ''; dist = -3
        elif gt_val == '':
            pred_val = _safe(r['pred_flat'].get(field, ''))
            dist = -1
        else:
            pred_val = _safe(r['pred_flat'].get(field, ''))
            dist = -2 if pred_val in ('None', '') else Levenshtein.distance(pred_val, gt_val)

        records.append({
            'image'       : r['image'],
            'doc_type'    : r['doc_type'],
            'entity'      : field,          
            'label_val'   : gt_val,
            'response_val': pred_val,
            'dist'        : dist,
            'valid_json'  : r['valid_json'],
            'model'       : CHECKPOINT,
            'pretty_name' : MODEL_NAME,
        })

df_multi = pd.DataFrame(records)
print(f"df_multi shape: {df_multi.shape}")
df_multi.head(3)

## 4 · Feature Type Analysis and Categorization

### 4.1 Null Percentage by Entity

In [ ]:
from plotnine import ggplot, aes, geom_bar, theme_bw, labs, coord_flip

null_df = (
    df_multi.groupby('entity')['label_val']
    .apply(lambda x: (x == '').sum() / len(x) * 100)
    .reset_index()
    .rename(columns={'label_val': 'metric_value'})
    .sort_values('metric_value', ascending=True)
)
null_df['entity'] = pd.Categorical(null_df['entity'], categories=null_df['entity'].tolist(), ordered=True)

(ggplot(null_df, aes(x='entity', y='metric_value')) +
 geom_bar(stat='identity', fill='steelblue') +
 coord_flip() +
 labs(title='Null Percentage by Entity', x='Entity', y='Null Percentage (%)') +
 theme_bw())

### 4.2 Entity String Length — Boxplot

In [ ]:
from plotnine import geom_boxplot, theme_minimal, theme

df_len = df_multi[df_multi['label_val'] != ''].copy()
df_len['str_len'] = df_len['label_val'].str.len()

entity_order = (
    df_len.groupby('entity')['str_len']
    .mean()
    .sort_values()
    .index.tolist()
)
df_len['entity'] = pd.Categorical(df_len['entity'], categories=entity_order, ordered=True)

(ggplot(df_len, aes(x='entity', y='str_len')) +
 geom_boxplot(fill='#abd9e9', color='#2c7bb6', outlier_alpha=0.3) +
 coord_flip() +
 labs(title='String Length Distribution by Entity', x='Entity', y='String Length') +
 theme_minimal() +
 theme(figure_size=(10, 6)))

### 4.3 Categorize Features

In [ ]:
NULL_THRESHOLD   = 70
LENGTH_THRESHOLD = 50

from enum import Enum

class FeatureCategory(str, Enum):
    MISSING_GROUND_TRUTH = 'missing_ground_truth'
    SHORT_TEXT           = 'short_text'
    LONG_TEXT            = 'long_text'

stats_records = []
for entity, group in df_multi.groupby('entity'):
    labels = group['label_val']
    null_pct = (labels == '').sum() / len(labels) * 100
    
    non_null = labels[labels != '']
    len_mean = non_null.str.len().mean() if len(non_null) > 0 else 0.0
    
    stats_records.append({
        'entity': entity,
        'null_pct': null_pct,
        'len_mean': len_mean
    })

entity_stats = pd.DataFrame(stats_records)

def categorize(row):
    if row['null_pct'] > NULL_THRESHOLD:
        return FeatureCategory.MISSING_GROUND_TRUTH
    elif row['len_mean'] <= LENGTH_THRESHOLD:
        return FeatureCategory.SHORT_TEXT
    else:
        return FeatureCategory.LONG_TEXT

entity_stats['category'] = entity_stats.apply(categorize, axis=1)
feature_categories = dict(zip(entity_stats['entity'], entity_stats['category']))

for cat in FeatureCategory:
    members = [e for e, c in feature_categories.items() if c == cat]
    print(f"\n{cat.value} ({len(members)}): {members}")

## 5 · Edit Distance Heatmap (All Entities)

In [ ]:
from plotnine import *
import pandas as pd

df = df_multi.copy()

df["dist_cut"] = df["dist"].clip(upper=6)
df["dist_cut"] = df["dist_cut"].replace(-2, 7)
df["dist_cut"] = df["dist_cut"].replace(-3, 8)
df["dist_cut"] = df["dist_cut"].replace(-4, 9)

mapper = {
    -1: "missing groundtruth",
     0: "exact match",
     1: "1", 2: "2", 3: "3", 4: "4", 5: "5", 6: "6+",
     7: "predicted 'None'",
     8: "key missing",
     9: "invalid JSON"
}

df["dist_cut"] = df["dist_cut"].replace(mapper)
df["dist_cut"] = pd.Categorical(df["dist_cut"], categories=list(mapper.values()), ordered=True)
df["entity_type"] = df["entity"].map(feature_categories)

# Thêm doc_type prefix
df["entity_label"] = df["doc_type"].fillna("UNKNOWN") + " | " + df["entity"]

df_sorted = df.sort_values(["doc_type", "entity_type", "entity"])
df_entities = df_sorted[["entity_label", "entity_type", "doc_type"]].drop_duplicates(subset=["entity_label"]).reset_index(drop=True)
ordered_values = list(df_entities["entity_label"])

change_indices = df_entities.index[
    (df_entities["entity_type"] != df_entities["entity_type"].shift()) |
    (df_entities["doc_type"] != df_entities["doc_type"].shift())
].tolist()
feature_class_lines = [i + 0.5 for i in change_indices[1:]]

df_sorted['pretty_name_ordered'] = pd.Categorical(
    df_sorted['pretty_name'],
    categories=sorted(df_sorted['pretty_name'].unique()),
    ordered=True
)

def plot_bar_chart_all_entities(df, ordered_values, width=22, height=18, geom_vlines=[]):
    custom_colors = ["#bababa","#66c2a5","#abdda4","#e6f598","#ffffbf",
                     "#fee08b","#fdae61","#f46d43","#abd9e9","#74add1","#bebada"]
    return (
        ggplot(df, aes(x="entity_label", fill="factor(dist_cut)"))
        + geom_bar(position="stack", color="black")
        + facet_wrap("~pretty_name_ordered", scales="free")
        + labs(x="Doc Type | Field", y="Count", fill="Char. edit distance")
        + coord_flip()
        + scale_fill_manual(values=custom_colors, labels=list(mapper.values()))
        + theme_minimal()
        + theme(figure_size=(width, height))
        + scale_x_discrete(limits=ordered_values)
    )

In [ ]:
from IPython.display import Markdown, display

plot = plot_bar_chart_all_entities(df_sorted, ordered_values, width=15, height=14, geom_vlines=feature_class_lines)
display(Markdown("### All Entities"))
plot.show()

## 6 · Model Performance Metrics (Exact Match, CER, BLEU, ROUGE)


In [ ]:
import evaluate

eval_metrics = ["exact_match", "character", "bleu", "rouge"]
evaluations  = [evaluate.load(m) for m in eval_metrics]

def calculate_metrics(references, predictions):

    pairs = [(r, p) for r, p in zip(references, predictions) if r.strip() != ""]
    
    if not pairs:
        return pd.Series({'exact_match': 0.0, 'cer': None, 'bleu': None, 'rouge': None})
    
    refs, preds = zip(*pairs)
    refs  = list(refs)
    preds = list(preds)
    

    results = {}
    for metric in evaluations:
        res = metric.compute(predictions=preds, references=refs)
        results.update(res)
    return results


In [ ]:
# Filter out MISSING_GROUND_TRUTH entities
relevant_entities = [
    e for e, cat in feature_categories.items()
    if cat != FeatureCategory.MISSING_GROUND_TRUTH
]
df_filtered = df_multi[df_multi['entity'].isin(relevant_entities)].copy()

df_eval = (
    df_filtered
    .groupby(['model', 'pretty_name', 'entity'])
    .apply(
        lambda x: calculate_metrics(
            x['label_val'].astype(str).tolist(),
            x['response_val'].astype(str).tolist()
        ),
        include_groups=False   # ← thêm dòng này
    )
    .reset_index()
)
df_eval = pd.concat([df_eval, df_eval[0].apply(pd.Series)], axis=1).drop(0, axis=1)
print(f'df_eval shape: {df_eval.shape}')
df_eval.head(3)

### 6.1 · Exact Match per Entity


In [ ]:
from plotnine import *

ordered_relevant = [e for e in relevant_entities]  # dùng entity gốc, không prefix

df_entities_rel = df_entities[df_entities['entity_label'].apply(
    lambda x: x.split(' | ')[-1] if ' | ' in x else x
).isin(relevant_entities)].reset_index(drop=True)

change_idx_rel = df_entities_rel.index[
    df_entities_rel['entity_type'] != df_entities_rel['entity_type'].shift()
].tolist()
feat_lines_rel = [i + 0.5 for i in change_idx_rel[1:]]

plot_em = (
    ggplot(df_eval, aes(x='entity', y='exact_match', fill='pretty_name'))
    + geom_bar(stat='identity', position='dodge', colour='gray')
    + labs(title='Exact Match per Entity (higher = better)',
           x='Entity', y='Exact Match', fill='Model')
    + coord_flip()
    + scale_fill_brewer(type='qual', palette='Set3')
    + theme_minimal()
    + theme(figure_size=(12, 8), axis_text_x=element_text(angle=45, hjust=1))
    + scale_y_continuous(
        breaks=[x/100 for x in range(0, 101, 10)],
        labels=[f'{i}%' for i in range(0, 101, 10)]
    )
    + geom_hline(yintercept=[x/100 for x in range(0, 101, 10)],
                 color='darkgray', size=0.4, alpha=0.5)
    + scale_x_discrete(limits=ordered_relevant)
)
plot_em

### 6.2 · Character Error Rate (CER) per Entity


In [ ]:
plot_cer = (
    ggplot(df_eval, aes(x='entity', y='cer_score', fill='pretty_name'))
    + geom_bar(stat='identity', position='dodge', color='gray')
    + labs(title='CER per Entity (lower = better)',
           x='Entity', y='CER', fill='Model')
    + coord_flip()
    + scale_fill_brewer(type='qual', palette='Set3')
    + theme_minimal()
    + theme(figure_size=(12, 8), axis_text_x=element_text(angle=45, hjust=1))
    + geom_vline(xintercept=feat_lines_rel, linetype='dashed',
                 color='#4d4d4d', size=2.0)
    + scale_x_discrete(limits=ordered_relevant)
)
plot_cer


### 6.3 · Aggregate Metrics Table

- **Short text** entities → Exact Match + CER  
- **Long text** entities → ROUGE scores


In [ ]:
import numpy as np

# ── Short-text aggregate (Exact Match + CER) ───────────────────────────
short_entities = [e for e, c in feature_categories.items()
                  if c == FeatureCategory.SHORT_TEXT]
exact_agg = (
    df_eval[df_eval['entity'].isin(short_entities)]
    .groupby(['model', 'pretty_name'])
    .agg({'exact_match': 'mean', 'cer_score': 'mean'})
    .reset_index()
    .rename(columns={'exact_match': 'accuracy (exact match)'})
)

# ── Long-text aggregate (ROUGE) ─────────────────────────────────────────
long_entities = [e for e, c in feature_categories.items()
                 if c == FeatureCategory.LONG_TEXT]
rouge_agg = (
    df_eval[df_eval['entity'].isin(long_entities)]
    .groupby(['model', 'pretty_name'])
    .agg({'rouge1': 'mean', 'rouge2': 'mean', 'rougeL': 'mean', 'rougeLsum': 'mean'})
    .reset_index()
)

aggregated_metrics = rouge_agg.merge(exact_agg, on=['model', 'pretty_name'], how='inner')

metric_cols = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum', 'accuracy (exact match)', 'cer_score']
aggregated_metrics[metric_cols] = aggregated_metrics[metric_cols].round(3)

def highlight_max(s, props=''):
    return np.where(s == np.nanmax(s.values), props, '')
def highlight_min(s, props=''):
    return np.where(s == np.nanmin(s.values), props, '')

higher_better = ['rouge1','rouge2','rougeL','rougeLsum','accuracy (exact match)']
lower_better  = ['cer_score']

aggregated_metrics.style \
    .apply(highlight_max, props='background-color:#99d594;', axis=0, subset=higher_better) \
    .apply(highlight_min, props='background-color:#99d594;', axis=0, subset=lower_better)


## 7 · Visual Diff — Prediction vs Ground Truth

Displays the document image alongside a colour-coded diff between
the predicted JSON and the ground-truth JSON for the first `N_SHOW` samples.


In [ ]:
import textwrap
from PIL import Image
import matplotlib.pyplot as plt

N_SHOW = 15
COL_W = 35  # độ rộng cột GT và Pred

for i, r in enumerate(rows[:N_SHOW]):
    gt_flat   = flatten_dict(r['gt'])
    pred_flat = flatten_dict(r['pred']) if r['valid_json'] else {}

    # Hiện ảnh
    p = Path(r['image'])
    if not p.is_absolute(): p = IMAGES_DIR / p.name
    try:
        img = Image.open(p).convert('RGB')
        plt.figure(figsize=(5, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"#{i+1} | {r['doc_type']}")
        plt.show()
    except Exception as e:
        print(f"⚠ Không load được ảnh: {e}")

    # Print diff
    status = '✓ valid JSON' if r['valid_json'] else '✗ invalid JSON'
    print(f"{status}")
    print(f"{'Field':<40} {'GT':<{COL_W}} Pred")
    print("-" * (40 + COL_W * 2 + 2))

    for k in sorted(gt_flat.keys()):
        gv = str(gt_flat.get(k, '')).strip()
        pv = str(pred_flat.get(k, '')).strip()
        ok = '✓' if gv == pv else '✗'


        gv_lines = textwrap.wrap(gv, COL_W) or ['']
        pv_lines = textwrap.wrap(pv, COL_W) or ['']
        n_lines  = max(len(gv_lines), len(pv_lines))

        for j in range(n_lines):
            prefix = f"{ok} {k:<38}" if j == 0 else f"  {'':<38}"
            g = gv_lines[j] if j < len(gv_lines) else ''
            p_ = pv_lines[j] if j < len(pv_lines) else ''
            print(f"{prefix} {g:<{COL_W}} {p_}")

    print()